# VascuQuest Parameterized Cohort Manual Qualification

This notebook is the manual, checkpointed real-PWDB qualification route for PR #20.

Run in order:

1. mount Drive and clone the PR #20 qualification branch;
2. stage/verify canonical PWDB artifacts to Colab SSD;
3. run **1 subject × 4 diseases** smoke qualification;
4. only after PASS, run **3 subjects × 4 diseases**;
5. finalize `parameterized-cohort-qualification.json`.

Every completed subject is written to Drive and verified immediately. A PASS remains **MODELLED**, not clinical validation.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os, shutil, subprocess, sys, json

REPO_URL = "https://github.com/KNOWDYN/VascuQuest.git"
QUALIFICATION_REF = "release/parameterized-cohort-qualification"

DRIVE_PWDB_ROOT = Path("/content/drive/MyDrive/VascuQuest/PWDB")
OUTPUT_ROOT = Path("/content/drive/MyDrive/VascuQuest/parameterized_cohort_qualification")
LOCAL_REPO = Path("/content/VascuQuest-qualification")
LOCAL_SOURCE = Path("/content/vascuquest-pwdb-source")
LOCAL_XDG = Path("/content/vascuquest-xdg")

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_SOURCE.mkdir(parents=True, exist_ok=True)
LOCAL_XDG.mkdir(parents=True, exist_ok=True)

os.environ["XDG_DATA_HOME"] = str(LOCAL_XDG / "data")
os.environ["XDG_CACHE_HOME"] = str(LOCAL_XDG / "cache")
os.environ["XDG_STATE_HOME"] = str(LOCAL_XDG / "state")

if LOCAL_REPO.exists():
    shutil.rmtree(LOCAL_REPO)

subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", QUALIFICATION_REF, REPO_URL, str(LOCAL_REPO)],
    check=True,
)
CODE_REVISION = subprocess.check_output(
    ["git", "-C", str(LOCAL_REPO), "rev-parse", "HEAD"], text=True
).strip()

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(LOCAL_REPO)], check=True)

RUNNER = LOCAL_REPO / "tests/full_data/parameterized_cohort_colab_validation.py"
print("Code revision:", CODE_REVISION)
print("PWDB root:", DRIVE_PWDB_ROOT)
print("Output root:", OUTPUT_ROOT)


## Phase A — rapid smoke qualification

This must pass before Phase B.

In [ ]:
subprocess.run(
    [
        sys.executable, str(RUNNER),
        "--phase", "smoke",
        "--drive-pwdb-root", str(DRIVE_PWDB_ROOT),
        "--local-source", str(LOCAL_SOURCE),
        "--output-root", str(OUTPUT_ROOT),
        "--code-revision", CODE_REVISION,
    ],
    check=True,
)
smoke = json.loads((OUTPUT_ROOT / "smoke_results.json").read_text())
print(json.dumps({"status": smoke["status"], "cases_completed": smoke["cases_completed"]}, indent=2))


In [ ]:
smoke = json.loads((OUTPUT_ROOT / "smoke_results.json").read_text())
if smoke.get("status") != "PASS" or smoke.get("cases_completed") != 4:
    raise RuntimeError("Smoke qualification is not complete/PASS.")
print("Smoke gate: PASS — full qualification authorized.")


## Phase B — 3 subjects × 4 diseases

Completed subjects are checkpointed to Drive. Rerun this cell to resume.

In [ ]:
subprocess.run(
    [
        sys.executable, str(RUNNER),
        "--phase", "full",
        "--drive-pwdb-root", str(DRIVE_PWDB_ROOT),
        "--local-source", str(LOCAL_SOURCE),
        "--output-root", str(OUTPUT_ROOT),
        "--code-revision", CODE_REVISION,
    ],
    check=True,
)
full = json.loads((OUTPUT_ROOT / "full_results.json").read_text())
print(json.dumps({"status": full["status"], "cases_completed": full["cases_completed"]}, indent=2))


## Finalize machine-readable qualification evidence

In [ ]:
subprocess.run(
    [
        sys.executable, str(RUNNER),
        "--phase", "finalize",
        "--drive-pwdb-root", str(DRIVE_PWDB_ROOT),
        "--local-source", str(LOCAL_SOURCE),
        "--output-root", str(OUTPUT_ROOT),
        "--code-revision", CODE_REVISION,
    ],
    check=True,
)
report_path = OUTPUT_ROOT / "parameterized-cohort-qualification.json"
report = json.loads(report_path.read_text())
print(json.dumps({
    "status": report["status"],
    "code_revision": report["code_revision"],
    "report": str(report_path),
    "scientific_boundary": report["scientific_boundary"],
}, indent=2))


After the final cell prints `PASS`, use the durable report at:

`MyDrive/VascuQuest/parameterized_cohort_qualification/parameterized-cohort-qualification.json`

as the evidence for deciding whether PR #20 is ready to merge. A PASS does not imply clinical validation.
